# PDEep G1 — one network for forward, inverse and hybrid information

One independent A-Geo/C-Geo instance per objective, shared across3 PDEs ×8 fixed geometries ×3 conditioning modes. Same physical cases, priors, reference labels and splits. Explicit C3 field availability drives predictions; mode labels are reporting metadata. **Safe Run All is preview-only.**

In [1]:
# Setup configuration — used only in Colab. Local sessions need an installed project.
REPO_URL = "SET_YOUR_GITHUB_REPOSITORY_URL"
REPO_REF = "main"  # Prefer an immutable full commit SHA or release tag for research.
PROJECT_DIR = "/content/pde-inverse-research"

USE_GOOGLE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/PDEInverseResearch"
LOCAL_OUTPUT_DIR = "/content/pde-inverse-outputs"

# Geometry installation extra required for G1_smooth_v1.
ENABLE_GEOMETRY = True


In [2]:
# Automatic setup — advanced implementation (collapsed). Run this cell normally.
# Stdlib bootstrap is embedded because pdeep is absent in a fresh Colab runtime.
import sys, json
from pathlib import Path
try:
    from IPython import get_ipython
    _shell = get_ipython()
except ImportError:
    _shell = None
IS_COLAB = "google.colab" in sys.modules or type(_shell).__module__.startswith("google.colab")

def ensure_checkout(repo_url, repo_ref, project_dir, *, git_env=None):
    """Clone only absent destinations; existing worktrees are strictly read-only.

    Supports token-free HTTPS and Git-over-SSH URLs. External credential helpers
    or SSH agents handle private authentication. Raw Git output stays in memory.
    """
    import os, re, subprocess, tempfile
    from pathlib import Path
    from urllib.parse import urlsplit
    import sys
    if sys.version_info < (3, 10):
        raise RuntimeError('PDEep requires Python >=3.10; select a supported runtime before setup.')
    def identity(value):
        if not isinstance(value,str) or not value or any(c in value for c in '\n\r\0'):
            raise ValueError('Configure a token-free repository URL; credentials must not be embedded.')
        if value.startswith('git@'):
            match=re.fullmatch(r'git@([A-Za-z0-9.-]+):([A-Za-z0-9_./-]+)',value)
            if not match:raise ValueError('Use git@HOST:OWNER/REPO.git with an external SSH agent.')
            host,path=match.groups()
        else:
            url=urlsplit(value)
            if url.scheme not in {'https','ssh'} or not url.hostname or url.password or url.query or url.fragment:
                raise ValueError('Use token-free HTTPS or Git SSH; credential URLs, queries and fragments are rejected.')
            if (url.scheme=='https' and url.username is not None) or (url.scheme=='ssh' and url.username not in {None,'git'}):
                raise ValueError('Do not put credentials/user tokens in repository URLs; use an external credential helper or SSH agent.')
            host=url.hostname;path=url.path.lstrip('/')
            if url.port not in {None,22,443}:raise ValueError('Unsupported repository port.')
        path=path.rstrip('/').removesuffix('.git')
        if not path or len(path.split('/'))<2 or '..' in path.split('/'):
            raise ValueError('Configure REPO_URL with OWNER/REPO; no local or credential-bearing URL is accepted.')
        if host.lower()=='github.com':path=path.lower()
        return host.lower()+'/'+path
    expected=identity(repo_url)
    if not isinstance(repo_ref,str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_./-]*',repo_ref) or '..' in repo_ref:
        raise ValueError('REPO_REF must be a branch, tag or full commit SHA, not a command or revision expression.')
    destination=Path(project_dir).expanduser().resolve()
    env=dict(os.environ);env.update(git_env or {});env['GIT_TERMINAL_PROMPT']='0'
    def git(arguments, *, cwd=None, purpose='repository verification'):
        try:result=subprocess.run(['git',*arguments],cwd=cwd,env=env,capture_output=True,text=True,timeout=180)
        except FileNotFoundError:raise RuntimeError('Git is missing; install Git before repository setup.') from None
        except (OSError,subprocess.TimeoutExpired):raise RuntimeError('Git '+purpose+' could not complete; no credentials/output were logged.') from None
        if result.returncode:
            error=result.stderr.lower()
            if any(s in error for s in ('authentication','could not read username','publickey','permission denied')):
                raise RuntimeError('Git authentication failed. Configure an external credential helper or SSH agent with read access; do not put secrets in this notebook.')
            raise RuntimeError('Git '+purpose+' failed. Check repository existence, access and REPO_REF. Raw output is suppressed to protect credentials.')
        return result.stdout.strip()
    def head(path):return git(['rev-parse','--verify','HEAD^{commit}'],cwd=path)
    def resolved_ref(path):
        if re.fullmatch(r'[0-9a-fA-F]{40}|[0-9a-fA-F]{64}',repo_ref):wanted=repo_ref.lower()
        else:
            refs=[repo_ref,repo_ref+'^{}'] if repo_ref.startswith('refs/') else ['refs/heads/'+repo_ref,'refs/tags/'+repo_ref,'refs/tags/'+repo_ref+'^{}']
            lines=git(['ls-remote','--exit-code','origin',*refs],cwd=path,purpose='remote reference lookup')
            mapping=dict((ref,sha) for sha,ref in (line.split() for line in lines.splitlines()))
            if repo_ref.startswith('refs/'):wanted=mapping.get(repo_ref+'^{}',mapping.get(repo_ref))
            else:
                branch=mapping.get('refs/heads/'+repo_ref);tag=mapping.get('refs/tags/'+repo_ref+'^{}',mapping.get('refs/tags/'+repo_ref))
                if branch and tag and branch!=tag:raise ValueError('Branch/tag name is ambiguous; use refs/heads/... or refs/tags/....')
                wanted=branch or tag
            if not wanted:raise ValueError('Configured REPO_REF was not found on origin.')
        return wanted
    existed=destination.exists()
    if existed:
        if not destination.is_dir() or not (destination/'.git').exists():
            raise ValueError('PROJECT_DIR exists but is not a Git checkout; choose a separate empty directory. Nothing was overwritten.')
        top=Path(git(['rev-parse','--show-toplevel'],cwd=destination)).resolve()
        if top!=destination:raise ValueError('PROJECT_DIR must be the repository root, not a nested directory.')
        origin=git(['config','--get','remote.origin.url'],cwd=destination)
        if identity(origin)!=expected:raise ValueError('Existing checkout origin differs from REPO_URL. No files or remotes were changed.')
        if git(['status','--porcelain','--untracked-files=normal'],cwd=destination):
            raise ValueError('Existing checkout has local modifications/untracked files. Commit them yourself or choose a new PROJECT_DIR; no reset/checkout was performed.')
        commit=head(destination)
        wanted=resolved_ref(destination)
        if commit!=wanted:
            raise ValueError('Existing checkout is at '+commit+' but REPO_REF resolves to '+wanted+'. Choose a new PROJECT_DIR or update it explicitly; nothing was reset.')
    else:
        destination.parent.mkdir(parents=True,exist_ok=True)
        # A failed clone stays in its own scratch directory, not PROJECT_DIR.
        with tempfile.TemporaryDirectory(prefix='.pdeep-clone-',dir=destination.parent) as scratch:
            checkout=Path(scratch)/'checkout'
            git(['clone','--no-checkout','--',repo_url,str(checkout)],purpose='clone')
            wanted=resolved_ref(checkout)
            git(['fetch','--depth','1','origin',wanted],cwd=checkout,purpose='configured reference fetch')
            git(['checkout','--detach','FETCH_HEAD'],cwd=checkout,purpose='fresh checkout')
            commit=head(checkout)
            if commit!=wanted:raise RuntimeError('Fetched checkout differs from the resolved reference; setup stopped.')
            required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
            if any(not (checkout/p).is_file() for p in required):
                raise ValueError('Repository/ref lacks the expected PDEep structure or this Colab setup version; select a compatible commit.')
            os.rename(checkout,destination)
    required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
    if any(not (destination/p).is_file() for p in required):raise ValueError('Expected pyproject.toml, pdeep package/setup and scientific contract are missing.')
    return {'project_dir':str(destination),'repository_identity':expected,'requested_ref':repo_ref,'git_commit':commit,
            'existing_checkout':existed,'worktree_modified_by_setup':not existed,'credentials_recorded':False}

BOOTSTRAP = None
if IS_COLAB:
    print("1. Verifying repository and preparing dependencies before scientific project imports.")
    BOOTSTRAP = ensure_checkout(REPO_URL, REPO_REF, PROJECT_DIR)
    import runpy
    _reference_setup = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/smooth/bootstrap.py"))
    ENVIRONMENT_SETUP = _reference_setup["prepare_reference_environment"](BOOTSTRAP["project_dir"], is_colab=True)
    _reference_setup["install_project"](BOOTSTRAP["project_dir"], source_revision=BOOTSTRAP["git_commit"])
    print("Dependency preparation complete; no loaded binary was replaced. Existing numerical certification is resolved below; no automatic requalification.")
print("2. Scientific imports and resolved environment.")
from pdeep.colab_setup import finish_setup
SETUP = finish_setup(is_colab=IS_COLAB, bootstrap=BOOTSTRAP, use_drive=USE_GOOGLE_DRIVE,
    drive_root=DRIVE_ROOT, local_output_dir=LOCAL_OUTPUT_DIR)
SETUP_CONFIGURATION = (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR)
ROOT = Path(SETUP["project_dir"])
from IPython.display import display, Markdown
from pdeep.notebook_workflows import initialize
from pdeep.runtime import capabilities
print("Colab:", IS_COLAB, "| imported source:", ROOT / "src/pdeep")
print("Exact Git commit:", SETUP["git_commit"] or "unavailable: local project is not a Git checkout")
print("Python:", SETUP["runtime"]["python"])
print("Dependency versions:", json.dumps(SETUP["runtime"]["dependencies"], indent=2))
for c in capabilities(): print(c.device, "available="+str(c.available), c.detail)
if IS_COLAB: print("Validated CUDA:", SETUP["cuda"])
print("Local unified root:", Path(SETUP["local_output_dir"]) / "G1_unified")
print("Drive unified root:", None if not SETUP["drive_root"] else Path(SETUP["drive_root"]) / "G1_unified")
print(SETUP["message"])

from pdeep.smooth.bootstrap import environment_versions
print("Python executable:", sys.executable)
print("Installed / loaded dependencies:", json.dumps(environment_versions(), sort_keys=True))

assert all((ROOT / p).is_file() for p in ("src/pdeep/unified/workflow.py", "docs/g1_unified_experiment_v1.md", "notebooks/04_g1_unified.ipynb")), "Configured repository/ref lacks the G1 unified workflow; select its reviewed commit."


2. Scientific imports and resolved environment.
Colab: False | imported source: /Users/nicolasackermann/Documents/ChatGPT/PDEep/src/pdeep
Exact Git commit: 0bee7c0bf285e3bb2ab8a18cefae129f33306e8e
Python: 3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:47) [Clang 13.0.0 (clang-1300.0.29.30)]
Dependency versions: {
  "pdeep-research": "0.1.0",
  "torch": "2.15.0",
  "numpy": "2.5.4",
  "scipy": "1.18.1",
  "matplotlib": "3.11.2",
  "pytest": "9.2.0",
  "notebook": "7.6.5",
  "nbformat": "5.11.1",
  "nbclient": "0.11.0",
  "ipykernel": "7.4.0",
  "setuptools": "84.2.1"
}
cpu available=True arm
mps available=True MPS float64 unsupported; explicit float32 neural execution only
cuda available=False CUDA unavailable
Local unified root: /Users/nicolasackermann/Documents/ChatGPT/PDEep/artifacts/G1_unified
Drive unified root: None
Local session: no clone, install or Google Drive import/mount; local outputs only.
Python executable: /Users/nicolasackermann/Documents/ChatGPT/PDEep/.venv/bin/python


In [3]:
from pdeep.learning_diagnostics.certificates import resolve_certification, copy_certificate, SUPPLIED_ID
CERTIFICATE_ID = SUPPLIED_ID
CERTIFICATE_PATH = None  # Optional full issued JSON path; no upload-path assumption.
USE_EXISTING_LOCAL_QUALIFICATION = not IS_COLAB  # Explicit local policy: reuse existing, never recompute.
CERTIFICATE_CACHE = Path(SETUP["local_output_dir"]) / "certifications/G1_smooth_v1"

ISSUED_CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": CERTIFICATE_ID},
    configured_path=CERTIFICATE_PATH, drive_root=SETUP["drive_root"], cache_dir=CERTIFICATE_CACHE)
display(ISSUED_CERTIFICATION.summary())
CERTIFICATION = ISSUED_CERTIFICATION
if not IS_COLAB and USE_EXISTING_LOCAL_QUALIFICATION and not CERTIFICATION.verified:
    CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": None}, cache_dir=CERTIFICATE_CACHE)
    print("Explicit local policy: selecting an existing native-runtime qualification; issued Linux qualification remains incompatible here.")
    display(CERTIFICATION.summary())
CERTIFICATE = CERTIFICATION.record if CERTIFICATION.verified else None
if CERTIFICATE is None:
    print("Training blocked. Restore the certified runtime/settings or place the full issued JSON in the displayed cache/Drive directory. No requalification has been executed.")
if CERTIFICATION.verified and SETUP["drive_root"]:
    try:
        CERTIFICATE_DRIVE_PATH = copy_certificate(CERTIFICATION.location,
            Path(SETUP["drive_root"]) / "certifications/G1_smooth_v1")
        print("Full certificate byte-copy verified:", CERTIFICATE_DRIVE_PATH)
    except Exception as error:
        print("CERTIFICATE DRIVE COPY FAILED; verified local copy retained:", type(error).__name__, str(error))
PERSISTENCE = SETUP["persistence"]

{'Benchmark': 'G1_smooth_v1',
 'Certification': 'INCOMPATIBLE',
 'Certificate ID': '524b9aa89bb0baa3606cc47657d6ee31c95ef2d8a212e9c96b6a4ba43d33c711',
 'Certified PDEs': ['heat', 'poisson', 'wave'],
 'Certified geometries': ['annulus',
  'disk',
  'ellipse',
  'hole',
  'lshape',
  'polygon',
  'rectangle',
  'square'],
 'Certified mesh resolution': [0.02],
 'Source compatibility': [{'field': 'runtime_identity.physics.dependencies.numpy',
   'certified': '2.4.6',
   'current': '2.5.4'},
  {'field': 'runtime_identity.physics.dependencies.scipy',
   'certified': '1.16.3',
   'current': '1.18.1'},
  {'field': 'runtime_identity.physics.scientific_sha',
   'certified': 'cf0918f951f38317a813a9b4728dfae3efe984379e09e5d3a42827d7cf3f66db',
   'current': '76fe35493cf0dd3c8dc29e94b8df2a51b0102d65328bca7a74fdc110962b4093'},
  {'field': 'runtime_identity.platform',
   'certified': 'Linux/x86_64',
   'current': 'Darwin/arm64'},
  {'field': 'runtime_identity.python',
   'certified': '3.13.16',
   'cu

Explicit local policy: selecting an existing native-runtime qualification; issued Linux qualification remains incompatible here.


{'Benchmark': 'G1_smooth_v1',
 'Certification': 'VERIFIED',
 'Certificate ID': '23b063563c89ff6e299907404f5f920d4b791a7b1337cf411359c51e58c3656f',
 'Certified PDEs': ['heat', 'poisson', 'wave'],
 'Certified geometries': ['annulus',
  'disk',
  'ellipse',
  'hole',
  'lshape',
  'polygon',
  'rectangle',
  'square'],
 'Certified mesh resolution': [0.02],
 'Source compatibility': 'matching',
 'Location': '/Users/nicolasackermann/Documents/ChatGPT/PDEep/artifacts/certifications/G1_smooth_v1/23b063563c89ff6e299907404f5f920d4b791a7b1337cf411359c51e58c3656f.json',
 'Raw file SHA256': '1c327a21426ff249732da39cb8aff3592d12ff3cb2ff6ff58ab3ed1be0e2e173',
 'Qualification rerun required': 'No',
 'Qualification executed': False}

## 2. Shared configuration and immutable training horizon
Main target50,000 successful updates/model; optional extension100,000. The schedule and exact continuous-label plan are declared for100,000 **from initialization**, with250 warmup updates and cosine decay to10% of initial LR at100k. Stopping at3k/15k/50k preserves that schedule; continuing to100k is exact resume, not stretching a30k/50k scheduler. Preparing the100k label plan costs about13.5GB of training labels plus evaluation/assets. Physical IDs and random fields remain unchanged.

Fresh training is the main comparison policy. `finetune_inverse` is a separately labeled weights-only experiment with a newly initialized field encoder and new optimizer/scheduler; old inverse-only checkpoints are never silently interpreted as three-mode models. All three models get matched task/mode/observation/mask streams and independent parameter objects. C variants start with equal weights.

Mode mixture starts at30% forward,40% inverse,30% hybrid, independently balanced per task. Over60 updates (240 cases), every task sees3/4/3 mode exposures. Hybrid views include random/geometric-coefficient groups and whole wave u0/v0 masks;25% have no observations. Exact C3 coefficients only: arbitrary analytic functions, spatial samples and new equations are not claimed.

In [4]:
from pdeep.unified.config import DatasetConfig
from pdeep.unified_modes.workflow import configurations as research_configurations, preview
from pdeep.preparation.feasibility import configuration as feasibility_configuration
from pdeep.preparation.feasibility_training import configurations as feasibility_configurations
G1_MODE = "research"
G1_DATASET_PROFILE = "feasibility"  # feasibility (288 cases/3000 updates) | research (5376/100k horizon)
G1_DATA_WORKERS = 0  #0 auto CPU/RAM budget;1 debugging; explicit safe process count otherwise.
G1_REFERENCE_BLOCK_CASES = 2  # Full independent RHS columns; RAM budget enforced.
G1_REUSE_REFERENCE_MANIFEST = None  # Exact existing research manifest; no search or automatic re-solve.
G1_RUN_MODE = "train"
G1_MODELS = ["A_GEO/KIN_001", "C_GEO/FIELD_ONLY", "C_GEO/ALL_001"]
G1_DATASET_PRESET = "research"
G1_MODEL_CAPACITY = "large"  # medium remains available for reproduction.
G1_PDES = ["poisson", "heat", "wave"]
G1_GEOMETRIES = ["square", "rectangle", "lshape", "disk", "ellipse", "hole", "annulus", "polygon"]
G1_CASES_PER_UPDATE = 4
G1_QUERIES_PER_CASE = 4096
G1_QUERY_CHUNK_SIZE = 2048
G1_OBSERVATION_COUNTS = [8, 16, 32, 64, 128]
G1_TARGET_SUCCESSFUL_UPDATES = 50000
G1_TRAINING_HORIZON = 100000
G1_EXTENDED_TARGET = G1_TRAINING_HORIZON
G1_SCHEDULER_HORIZON = G1_TRAINING_HORIZON  # Declared BEFORE data preparation/training.
G1_MODES = ["FORWARD", "INVERSE", "HYBRID"]
G1_MODE_MIXTURE = (0.30, 0.40, 0.30)
G1_PREFLIGHT_QUERY_COUNTS = ()  # Optional2048/4096/8192 queries/case; needs exact prepared labels.
G1_STOP_AFTER_UPDATES = 50000  # Or3000,15000,100000 on the SAME declared horizon.
G1_LR = 3e-4
G1_WEIGHT_DECAY = 1e-4
G1_GRAD_CLIP = 1.0
G1_AMP = "auto"
G1_VALIDATE_EVERY = 500
G1_CHECKPOINT_EVERY = 500
G1_SEED = 42
G1_NEURAL_SEED = None  # Optional separate initialization seed; data streams stay fixed.
G1_CHECKPOINT_LOOKUP = "none"
G1_INITIALIZATION = "fresh"  # Keep ORIGINAL fresh/finetune_inverse on resume.
G1_ACTION = "fresh"  # fresh | resume | finetune_inverse; explicit parent paths for latter two.
G1_EXPERIMENT_ID = None  # Fresh UUID. For exact resume set the ORIGINAL displayed experiment ID.
G1_RESUME_PATHS = {}  # Exact manifest paths keyed by model, or load a known experiment.json below.
G1_DATA_ACTION = "prepare"  # preview | prepare | load | restore
G1_DATASET_MANIFEST = None  # Exact portable manifest/directory on local disk or Drive; .tar.gz also accepted for restore.
G1_PREPARE_MAX_CASES = None  # Optional1 bounds NEW case preparation for storage/Drive profiling.
G1_RUN_PREFLIGHT = False
G1_EVALUATE = False
G1_EVAL_SPLIT = "test"  # Explicitly choose test only after all tuning/checkpoint selection.
G1_EVAL_SELECTION = "best"
G1_BASELINE_ACTION = "preview"  # preview | prepare (CPU expensive) | evaluate (requires exact operators)
G1_CLASSICAL_INDEX = None  # Exact previously completed/resumable classical index, if available.
G1_EVAL_GRID_SIZE = 64
G1_EVAL_MAX_CASES_PER_TASK = None  # Optional bounded user pilot; report its restricted scope.
G1_EVAL_PHYSICS = True  # Existing separate weak/BC/IC/aux/wave diagnostics at largest count.
G1_PLOT_PHYSICAL_TIME = 0.2

if G1_DATASET_PROFILE == "feasibility" and G1_MODE != "smoke":
    G1_MODE = "pilot"
    G1_TARGET_SUCCESSFUL_UPDATES = G1_TRAINING_HORIZON = G1_EXTENDED_TARGET = G1_SCHEDULER_HORIZON = G1_STOP_AFTER_UPDATES = 3000
    G1_DATASET_PRESET = "pilot"
assert G1_CASES_PER_UPDATE == 4 and G1_OBSERVATION_COUNTS == [8,16,32,64,128]
if G1_MODE == "smoke":
    G1_DATASET_PRESET = "smoke"
    G1_TARGET_SUCCESSFUL_UPDATES, G1_TRAINING_HORIZON, G1_STOP_AFTER_UPDATES = 50, 50, 10
    G1_SCHEDULER_HORIZON = G1_TRAINING_HORIZON
    G1_QUERIES_PER_CASE, G1_QUERY_CHUNK_SIZE = 128, 64
G1_DATA_CONFIG = DatasetConfig(preset=G1_DATASET_PRESET, pdes=tuple(G1_PDES), families=tuple(G1_GEOMETRIES), seed=G1_SEED,
    stream_updates=G1_SCHEDULER_HORIZON, stream_queries=G1_QUERIES_PER_CASE)
if G1_DATASET_PROFILE == "feasibility" and G1_MODE != "smoke":
    G1_DATA_CONFIG = feasibility_configuration(seed=G1_SEED)
    configurations = feasibility_configurations
else:
    configurations = research_configurations
G1_ROOT = Path(SETUP["local_output_dir"]) / "G1_unified"
G1_CONFIGS = configurations(G1_DATA_CONFIG, models=G1_MODELS, experiment_id=G1_EXPERIMENT_ID,
    device=SETUP["training_device"], capacity=G1_MODEL_CAPACITY, updates=G1_SCHEDULER_HORIZON, main_target=G1_TARGET_SUCCESSFUL_UPDATES,
    query_count=G1_QUERIES_PER_CASE, query_chunk=G1_QUERY_CHUNK_SIZE, seed=G1_SEED,
    learning_rate=G1_LR, weight_decay=G1_WEIGHT_DECAY, grad_clip=G1_GRAD_CLIP,
    validate_every=G1_VALIDATE_EVERY, checkpoint_every=G1_CHECKPOINT_EVERY, amp=G1_AMP, mode=G1_MODE, neural_seed=G1_NEURAL_SEED,
    mode_mixture=G1_MODE_MIXTURE, initialization="finetune_inverse" if G1_ACTION == "finetune_inverse" else G1_INITIALIZATION)
# Resume requires the original initialization policy too. For a previously fine-tuned
# run, pass initialization="finetune_inverse" above while selecting action="resume".
if G1_ACTION == "resume" and G1_RESUME_PATHS:
    from pdeep.unified_modes.compatibility import resume_configurations
    G1_CONFIGS = resume_configurations(G1_CONFIGS, G1_RESUME_PATHS)
G1_EXPERIMENT_ID = G1_CONFIGS[0].experiment_id
G1_PREFLIGHT_PATH = None
G1_DATASET = None
G1_RESULT = None
display(preview(G1_DATA_CONFIG, G1_CONFIGS, G1_ROOT))
print("Experiment ID:", G1_EXPERIMENT_ID)
print("Session stop:", G1_STOP_AFTER_UPDATES, "on the declared", G1_TARGET_SUCCESSFUL_UPDATES, "main target; immutable scheduler horizon:", G1_SCHEDULER_HORIZON)

{'dataset': {'preset': 'pilot',
  'pdes': ('poisson', 'heat', 'wave'),
  'families': ('square',
   'rectangle',
   'lshape',
   'disk',
   'ellipse',
   'hole',
   'annulus',
   'polygon'),
  'counts': (8, 2, 2),
  'conditioning_h': 0.12,
  'seed': 42,
  'fine_spot_checks_per_task': 1,
  'qc_points': 128,
  'reference_tolerance': 0.002,
  'cache_tolerance': 0.0002,
  'response_relative_tolerance': 1e-06,
  'reference_execution_source': 'bc78e91eeece4c3c0e2ac4df38850e6a2150dbdfbe79ca29ed6f9fbf5eebd280',
  'stream_updates': 3000,
  'stream_queries': 4096,
  'stream_validation_queries': 512,
  'stream_grid': 64,
  'holdout_family': None,
  'schema': 'G1-unified-multicase-v1',
  'identity': 'ec1b17c1d2b5d223814901034e1ad8428f36bf018de6d3d17e01dbaa03ec8653',
  'tasks': 24,
  'totals': {'train': 192, 'validation': 48, 'test': 48},
  'scope': 'unseen physical fields on eight familiar fixed polygon instances; NOT geometry transfer',
  'prior_calibration': 'existing frozen TRAIN-only normalizer

Experiment ID: g1-modes-3ddab5c8772a49dd
Session stop: 3000 on the declared 3000 main target; immutable scheduler horizon: 3000


## 3. Prepare once; verify, copy and reuse
Default remains preview. Set `G1_DATA_ACTION="prepare"` explicitly for CPU64 generation on Mac or Colab. New self-contained folder: `G1_ROOT/datasets/<dataset_identity>/manifest.json`, with relative case/cache/certificate paths. All modes use the SAME targets. No solver runs in training.

Preparation reuses assembled original FEM operators/LUs, all32 full Poisson source responses, exact P4 values-only sampling and batched continuous queries. Heat CN/wave Gauss4,dt=.002 and original C1 output are unchanged. Independent per-case and stratified finer checks stay active. Completed artifacts and per-case label journals survive interruption. Phase timings, exact unique design counts, per-task throughput and measured ETA are reported; unmeasured tasks have no invented ETA. For continuation, select the exact existing portable manifest and `prepare` again; configuration and producer certificate must match.

**Move the whole dataset folder, never just manifest.json.** `load` selects an exact local folder/manifest. `restore` selects an exact Drive folder/manifest or verified .tar.gz+sidecar and copies to fast local Colab storage, verifies checksums, then loads. Training does not repeatedly read reference files from Drive. Dataset and producer certificate provenance remain unchanged across qualified Mac/Linux consumers; generation and consumer numerical runtimes still require their own existing native certificates. No automatic qualification.

Drive destination: `MyDrive/PDEInverseResearch/G1_unified/datasets/<dataset_identity>/`. Filesystem copying is sufficient; no Drive API or recursive discovery. Colab preparation periodically syncs verified files and manifest; local-save and sync failures are distinct, and valid local data are retained. An incomplete folder is never labeled complete.

For a legacy `datasets/<id>/index.json`, first run the explicit `python -m pdeep.preparation export-legacy` command with its original producer certificate. This copies completed records without solving or changing them. Then resume the exported manifest. Historical datasets/checkpoints remain untouched. See `docs/g1_dataset_preparation.md` for exact Mac/Drive/Colab commands and optional verified archive transfer.

**Balanced feasibility:** default profile is288 cases (8 TRAIN/2 validation/2 test per each24 tasks), all3 modes and3000 successful updates on a separately declared3000 horizon. It is a pipeline/learning feasibility study, not a full generalization claim. Research remains a separate5376-case/100k-plan folder; no existing research checkpoint schedule is changed.

Set `G1_REUSE_REFERENCE_MANIFEST` to the exact research manifest to derive existing matching physical cases from unique parent-counter draws without solving them again. New logical visits map to unused independent reference visits; this is an explicitly versioned sampling view, not recycled labels. Missing donor designs block reuse. The physical IDs and original split namespaces are unchanged; expanding a cohort uses a new experiment/manifest.

CPU processes use spawn, one native numerical thread per worker, bounded queues, per-worker FEM/evolution caches and one manifest coordinator. Automatic workers depend on RAM as well as CPU;4 workers can be unsafe for P4. Resource policy, active workers, aggregate sampled RSS, all24 task counts, throughput and measured ETA stay visible. No expensive generation starts until data action prepare is explicitly selected.

Feasibility model selection uses the existing validation reports every500 updates. Keep its TEST prefix untouched while making architecture/hyperparameter decisions, so expansion into full research does not contaminate final test evaluation. An expanded case pool is a new manifest/experiment; existing cases retain their original split. Exact donor labels are reused where present. Extending the requested coordinate set beyond stored labels is explicitly blocked rather than silently re-solving completed donor cases.

Read the current research index without generation in another idle kernel/cell:
```python
from pdeep.preparation.status import status
from pdeep.preparation.feasibility import parent_configuration
RESEARCH_MANIFEST = G1_ROOT / "datasets" / parent_configuration(feasibility_configuration(seed=G1_SEED)).identity / "manifest.json"
print(RESEARCH_MANIFEST)
display(status(RESEARCH_MANIFEST, verify="index"))  # fast, recorded counts
# Optional full completed-record audit: verify="checksums" or "payloads".
```

In [6]:
from pdeep.preparation.workflow import prepare
if G1_DATA_ACTION != "preview":
    CERTIFICATION.require()
    G1_DATASET = prepare(G1_DATA_CONFIG, G1_ROOT, CERTIFICATE, action=G1_DATA_ACTION,
        manifest=G1_DATASET_MANIFEST, persistence=PERSISTENCE, max_cases=G1_PREPARE_MAX_CASES,
        workers=G1_DATA_WORKERS, block_cases=G1_REFERENCE_BLOCK_CASES,
        feasibility=G1_DATASET_PROFILE == "feasibility" and G1_MODE != "smoke", reuse_from=G1_REUSE_REFERENCE_MANIFEST)
    G1_DATASET_MANIFEST = str(G1_DATASET.path)
    display(G1_DATASET.summary())
else:
    print("Dataset preview only: no reference preparation, dataset loading or qualification executed.")

{'dataset_identity': 'ec1b17c1d2b5d223814901034e1ad8428f36bf018de6d3d17e01dbaa03ec8653',
 'target': {'train': 192, 'validation': 48, 'test': 48},
 'prepared': 288,
 'quarantined': 0,
 'stratified_checks': {'heat/annulus': {'cases': ['d3ff13a09b7b7cb6b2ae4fd3405f05681bf9908f4feadbfb12bab0e7985863cb'],
   'passed': True,
   'policy': 'unchanged independent finer controls'},
  'heat/disk': {'cases': ['dc9ac3e2173c1c9dcff042310a016e912ca699de8ccf718047a1de7ea56d83f4'],
   'passed': True,
   'policy': 'unchanged independent finer controls'},
  'heat/ellipse': {'cases': ['e42d45d5fd56fe112e501b7a2dedfe007358cf2e58a739a528cedd9603af6edc'],
   'passed': True,
   'policy': 'unchanged independent finer controls'},
  'heat/hole': {'cases': ['ddd2ae689b6b088e132207e40c9aaa9a4ad66264e0f454ceca9cf846965253dc'],
   'passed': True,
   'policy': 'unchanged independent finer controls'},
  'heat/lshape': {'cases': ['b0878acf5f1bfa0e861da545891472483abbfe8d50ab293c50598973751ee9d6'],
   'passed': True,
  

## 4. Architecture inventory and bounded real backward preflight
Large uses width384,8 heads,2 graph/geometry/observation/memory layers and4 decoder layers. Actual instantiated counts must exceed20M. Medium retains its original dimensions and weights. The inventory lists every component, token caps and FP32 weight/gradient/Adam memory before training; this is not measured activation memory.

Check27 model/PDE/mode combinations at the configured query count, then a real four-independent-case accumulated backward per large model with Forward/Inverse/Hybrid, all PDEs and irregular wave domains. Each backward contributes one quarter of its unchanged objective. Report allocated/reserved peaks, conservative lazy-Adam allowance, data/forward/backward/scaling timing, physics-call timing and throughput. There are no optimizer steps. CUDA must fit11.5GB or80% of actual device memory, whichever is smaller; an OOM blocks training without reducing queries or effective batch. CPU measurements do not qualify T4. Actual T4 feasibility is unverified until this cell passes there.

Execution already reuses case context across checkpointed decoder chunks and caches immutable geometry/graph tensors only. FP32 is the verified path: auto does not enable unverified FP16/BF16. Optional throughput candidates require prepared exact labels. No artificial allocation to meet a utilization target.

In [ ]:
from pdeep.unified_modes.preflight import inventory, run_preflight
if CERTIFICATE is not None:
    from pdeep.smooth.certification import normalizers
    display(inventory(G1_CONFIGS, normalizers(CERTIFICATE)))
if G1_RUN_PREFLIGHT:
    assert G1_DATASET is not None, "Prepare/load the dataset first."
    G1_PREFLIGHT_PATH, G1_PREFLIGHT = run_preflight(G1_CONFIGS, G1_DATASET, G1_ROOT, query_counts=G1_PREFLIGHT_QUERY_COUNTS)
    display({"path": G1_PREFLIGHT_PATH, "passed": G1_PREFLIGHT["passed"], "seconds": G1_PREFLIGHT["seconds"], "optimizer_steps": 0,
             "four_case_measurements": [{k:v for k,v in r.items() if k not in ("active_parameter_names", "inventory")} for r in G1_PREFLIGHT["accumulation_reports"]]})
else:
    print("Preflight disabled. Research training remains gated.")

## 5. One sequential training cell
Select `G1_RUN_MODE="train"`, resolve configuration, load the prepared data and run preflight first. One call trains A_GEO/KIN_001, C_GEO/FIELD_ONLY and C_GEO/ALL_001 sequentially. Each learns all modes and tasks; there is no mode-specific network. The unchanged successful-update AdamW/accumulation/clipping loop performs exactly one step/four physical cases.

Feasibility defaults to3000 successful updates on its own3000-update data/scheduler plan. Research keeps main stop50k and milestones3k/15k/50k/100k on the original100k plan. Save cadence500 includes each milestone. Never change the saved horizon on exact resume. Use the ORIGINAL experiment ID, initialization policy and scientific settings, `G1_ACTION="resume"`, lookup `"explicit"`, exact paths and a new absolute stop. A completed model is retained. Fine-tuning uses `G1_ACTION="finetune_inverse"`, exact inverse-only parents and a NEW experiment ID; old optimizer/scheduler/counters are not restored.

The existing single dashboard now also shows actual mode/PDE reconstruction means with physical-case denominators, cumulative72-category exposure, observation counts and known fractions. Raw/weighted losses, inactive N/A, wave-only kinematic scope, measured median ETA after10 updates, hardware, validation and local/Drive states remain visible. Unstarted architectures have no invented ETA. Known coefficients are clamped in auxiliary physical inputs; hidden source/initial truth is only the explicitly supervised auxiliary target.

Checkpoints verify locally first, sync periodically using the existing system and preserve local data on Drive failure. Reload/restart the kernel after source updates; do not hot-reload an active process.

For50k/100k scalability, the existing run loop executes in segments at validation/milestone boundaries, releasing its per-call report list each time. It never resets optimizer, scheduler or sampler. Verified incremental history shards and retained rotated local logs bound checkpoint log-scanning work; no historical artifacts are overwritten.

Large defaults to **fresh** initialization. Medium/old256-wide large weights cannot resume384-wide large. Both C losses use identical architecture sizes but separate trainable instances and separate measured ETA. Existing medium checkpoints retain full shape/source/configuration checks through a narrowly reviewed compatibility transition. Architecture expansion is not supported. Checkpoints contain exact architecture configuration and measured parameter counts.

In [ ]:
from pdeep.preparation.feasibility_training import train_models
G1_RESULT = train_models(G1_CONFIGS, G1_DATASET, G1_ROOT, run_mode=G1_RUN_MODE,
    stop_after=G1_STOP_AFTER_UPDATES, lookup=G1_CHECKPOINT_LOOKUP, action=G1_ACTION,
    resume_paths=G1_RESUME_PATHS, persistence=PERSISTENCE, preflight_path=G1_PREFLIGHT_PATH)
if G1_RUN_MODE == "preview":
    display(G1_RESULT)
else:
    display({key: {k:r[k] for k in ("status","successful","latest","best")} for key,r in G1_RESULT.items()})

## 6. Unseen forward/inverse/hybrid neural evaluation
Enable `G1_EVALUATE` only after validation-based selection. Exact checkpoint paths only; no optimizer or Drive search. Forward has all physical inputs and zero observations. Inverse hides them and sweeps8/16/32/64/128 sensors. Hybrid sweeps25/50/75% known,0/8/16/32/64/128 sensors, random/group masks and wave u0-known/v0-known patterns. The same case, reference queries and noisy observations are shared across models. Results retain mode, quantity, known-pattern/fraction, observation and query identities, time/late errors, physical RMSE and separate physical residuals. Neural predictions are approximate deterministic outputs, not certified solves or uncertainty samples.

In [ ]:
from pdeep.unified_modes.workflow import evaluate_checkpoints
G1_MODE_EVALUATION = None
G1_EVAL_PATHS = dict(G1_RESUME_PATHS)
if G1_RUN_MODE == "train" and G1_RESULT:
    G1_EVAL_PATHS.update({k:r["latest"] for k,r in G1_RESULT.items()})
if G1_EVALUATE and G1_DATASET_PROFILE == "feasibility":
    print("Feasibility selection uses the live fixed validation reports/checkpoints. Test sweep is reserved for finalized research; no feasibility test-driven tuning.")
elif G1_EVALUATE:
    assert G1_DATASET is not None, "Load the qualified dataset first."
    G1_MODE_EVALUATION = evaluate_checkpoints(G1_EVAL_PATHS, G1_CONFIGS, G1_DATASET, G1_ROOT,
        split=G1_EVAL_SPLIT, selection=G1_EVAL_SELECTION, max_cases_per_task=G1_EVAL_MAX_CASES_PER_TASK,
        grid_size=G1_EVAL_GRID_SIZE, diagnostics=G1_EVAL_PHYSICS, persistence=PERSISTENCE)
    display(G1_MODE_EVALUATION["report"]["main"])
else:
    print("Three-mode neural evaluation disabled; no test data or weights loaded.")

## 7. Validated numerical forward and Bayesian inverse/hybrid controls
Classical methods require no neural training. Explicit `G1_BASELINE_ACTION="prepare"` batches full P4/.02 CPU64 basis responses using the unchanged CN/Gauss4/.002 continuous reference. `"evaluate"` reuses verified matrices and completed case/view outputs. Current offsets are0; generic Gaussian algebra supports nonzero offsets and correlated priors. Hybrid first conditions the public Gaussian prior on exact known coefficients, then conditions the remaining coefficients on noisy observations. Full knowledge produces deterministic numerical propagation with exactly zero statistical variance. Partial knowledge plus zero sensors returns the conditional prior.

The original inverse-only prior/interpolation/Bayesian comparison remains available and is matched to the new networks' inverse rows. Interpolation respects holes and u/v channels. Official count designs remain reproducible non-nested layouts with shared keyed streams; independence across counts is not assumed. No noise floor or explicit inverse. Calibration describes latent fields separately from future measurement noise; numerical reference error is not posterior variance. Unknown noisy coefficient knowledge requires a separate likelihood and is not accepted as exact known information.

In [ ]:
from pdeep.unified_bayes.workflow import classical
from pdeep.unified_bayes.operators import ForwardOperators
from pdeep.unified_bayes.storage import ArtifactStore
from pdeep.unified_modes.classical import evaluate as evaluate_conditional_baselines
G1_CLASSICAL_RESULT = classical(G1_DATASET, G1_ROOT, action=G1_BASELINE_ACTION,
    counts=tuple(G1_OBSERVATION_COUNTS), size=G1_EVAL_GRID_SIZE,
    max_cases_per_task=G1_EVAL_MAX_CASES_PER_TASK, persistence=PERSISTENCE)
G1_CONDITIONAL_BASELINES = None
if G1_BASELINE_ACTION == "evaluate":
    G1_CLASSICAL_INDEX = G1_CLASSICAL_RESULT["index"]
    G1_BASELINE_ROOT = G1_ROOT / "baselines" / G1_DATASET.config.identity
    G1_BASELINE_STORE = ArtifactStore(G1_BASELINE_ROOT, PERSISTENCE)
    G1_OPERATORS = ForwardOperators(G1_BASELINE_ROOT / "operators", CERTIFICATE)
    G1_CONDITIONAL_BASELINES = evaluate_conditional_baselines(G1_DATASET, G1_OPERATORS,
        G1_BASELINE_ROOT / "conditioning-views", grid_size=G1_EVAL_GRID_SIZE,
        max_cases_per_task=G1_EVAL_MAX_CASES_PER_TASK, persist=G1_BASELINE_STORE)
    G1_BASELINE_STORE.index(G1_CONDITIONAL_BASELINES["index"])
display(G1_CLASSICAL_RESULT)

## 8. Compact comparison and recovery
The main table keeps forward, inverse and hybrid components separate. Task matrices cover24 tasks/mode; expanded cells retain known fraction/pattern, sensor count and physical time. Frozen TRAIN RMS defines standardized MSE. Physical RMSE/residuals stay separate. Confidence intervals resample physical cases, not pixels; paired comparisons verify identical information identities.

Inverse-only classical comparisons retain all six methods; forward uses numerical propagation, hybrid uses conditional prior/posterior controls. Missing model results are explicitly unavailable. No scientific accuracy claim exists before user-run test evaluation. Artifacts use existing verified local-first Drive sync and exact indices. Checkpoints stay under `DriveRoot/G1_unified/runs/<experiment>/<model>/...`; no recursive searches. After disconnect restore the exact data-plan index, original config/horizon/initialization, explicit checkpoint paths and rerun preflight. Optional `05_g1_playground.ipynb` loads a compatible checkpoint and performs inference only. See `docs/g1_conditioning_modes.md`.

Evaluation writes one verified bundle per completed physical case, containing all information views. This bounds Drive file/copy overhead; interrupted work resumes at the next incomplete case. Reports for different checkpoints/selections have separate identities and registries. Derived tables/PNGs are verified bundles referencing raw case indices, avoiding duplicate raw datasets.

In [ ]:
from pdeep.unified_modes.workflow import combined
from IPython.display import Image
if G1_MODE_EVALUATION:
    G1_MODE_COMPARISON = combined(G1_MODE_EVALUATION, G1_CONDITIONAL_BASELINES,
        G1_CONFIGS, G1_ROOT, inverse_index=G1_CLASSICAL_INDEX, persistence=PERSISTENCE)
    display(G1_MODE_COMPARISON["report"]["main"])
    display(G1_MODE_COMPARISON["report"]["forward_hybrid_baseline_comparison"])
    if "inverse_six_method" in G1_MODE_COMPARISON["report"]:
        display([r for r in G1_MODE_COMPARISON["report"]["inverse_six_method"]["tables"] if r["group"] == "overall"])
    display({"verified_summary":G1_MODE_COMPARISON["comparison_manifest"],
             "local_and_Drive":G1_MODE_COMPARISON["persistence"]})
    for path in G1_MODE_COMPARISON["plots"]:
        display(Image(filename=path))
else:
    print("No research comparison before explicit evaluation. Preview wrote no checkpoints.")
print("Exact Git revision:", SETUP["git_commit"])
print("Actual T4 and Drive verification requires Colab execution.")